In [11]:
import re
import json
import os
from datetime import datetime, timedelta

class AgentePersonalAvanzado:
    def __init__(self, archivo="actividades.json"):
        self.archivo = archivo
        self.actividades = []  # lista de diccionarios: nombre, inicio, duracion, prioridad, recurrencia, tipo
        self.cargar()

    def cargar(self):
        if os.path.exists(self.archivo):
            with open(self.archivo, "r") as f:
                try:
                    self.actividades = json.load(f)
                except json.JSONDecodeError: # Manejar archivo JSON vacío o malformado
                    self.actividades = []

    def guardar(self):
        with open(self.archivo, "w") as f:
            json.dump(self.actividades, f, indent=2)

    def limpiar_actividades(self):
        self.actividades = []
        if os.path.exists(self.archivo):
            os.remove(self.archivo)
        self.guardar() # Recrear un archivo vacío
        return "Todas las actividades han sido eliminadas."

    def eliminar_actividad(self, nombre_actividad):
        nombre_actividad_lower = nombre_actividad.lower()
        actividades_antes = len(self.actividades)
        self.actividades = [act for act in self.actividades if act['nombre'].lower() != nombre_actividad_lower]
        if len(self.actividades) < actividades_antes:
            self.guardar()
            return f"🗑️ Actividad(es) '{nombre_actividad}' eliminada(s)."
        else:
            return f"❌ No se encontró la actividad '{nombre_actividad}'."

    # --- Interpretación de lenguaje natural ---
    def extraer_fecha_hora(self, texto):
        ahora = datetime.now()
        fecha_resultante = ahora # Comenzar con la fecha y hora actuales

        # 1. Manejar "dentro de X horas/minutos" primero, ya que es relativo a 'ahora'.
        # Esto tiene prioridad sobre cualquier otro análisis de fecha/hora si está presente.
        tiempo_rel_match = re.search(r"dentro de (\d+)\s*(horas|minutos)", texto)
        if tiempo_rel_match:
            valor = int(tiempo_rel_match.group(1))
            unidad = tiempo_rel_match.group(2).lower()
            if unidad == "horas":
                return ahora + timedelta(hours=valor)
            elif unidad == "minutos":
                return ahora + timedelta(minutes=valor)

        # 2. Determinar el componente de FECHA (restablece la hora a 00:00 para fechas relativas).
        # La fecha explícita YYYY-MM-DD tiene prioridad si está presente, luego las palabras de día relativas.
        match_fecha_expl = re.search(r"(\d{4}-\d{2}-\d{2})", texto)
        if match_fecha_expl:
            try:
                parsed_date = datetime.strptime(match_fecha_expl.group(1), "%Y-%m-%d")
                fecha_resultante = fecha_resultante.replace(year=parsed_date.year, month=parsed_date.month, day=parsed_date.day, hour=0, minute=0, second=0, microsecond=0)
            except ValueError:
                pass # Si la fecha explícita está malformada, continuar con las palabras de día relativas
        elif "mañana" in texto:
            fecha_resultante = (ahora + timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)
        elif "pasado mañana" in texto:
            fecha_resultante = (ahora + timedelta(days=2)).replace(hour=0, minute=0, second=0, microsecond=0)
        elif "hoy" in texto: # Si 'hoy' está presente, establecer explícitamente al inicio del día
            fecha_resultante = ahora.replace(hour=0, minute=0, second=0, microsecond=0)
        # else: # Si no se menciona una fecha específica, y no es 'dentro de X', usar la fecha actual, pero mantener su hora actual
        #     fecha_resultante = ahora # Esta línea se maneja implícitamente por la inicial `fecha_resultante = ahora`

        # 3. Determinar el componente de HORA (HH:MM o HHam/pm)
        hora_encontrada = False

        hora_match_colon = re.search(r"(\d{1,2}):(\d{2})", texto)
        if hora_match_colon:
            h = int(hora_match_colon.group(1))
            m = int(hora_match_colon.group(2))
            fecha_resultante = fecha_resultante.replace(hour=h, minute=m, second=0, microsecond=0)
            hora_encontrada = True
        else:
            hora_match_ampm = re.search(r"(\d{1,2})\s*(am|pm)", texto, re.IGNORECASE)
            if hora_match_ampm:
                h = int(hora_match_ampm.group(1))
                ampm = hora_match_ampm.group(2).lower()
                if ampm == "pm" and h != 12:
                    h += 12
                elif ampm == "am" and h == 12: # 12 AM es 00:00
                    h = 0
                fecha_resultante = fecha_resultante.replace(hour=h, minute=0, second=0, microsecond=0)
                hora_encontrada = True

        # 4. Si no se encontró una hora específica (ni relativa ni explícita), por defecto 9:00 AM en la fecha determinada.
        # Si la fecha se estableció por 'hoy', 'mañana', 'pasado mañana' o YYYY-MM-DD explícito, su hora se restableció a 00:00,
        # así que aplicamos 09:00. Si no se encontraron palabras clave de fecha y no hay hora explícita, fecha_resultante es 'ahora',
        # así que reemplazamos su hora con 09:00.
        if not hora_encontrada:
            fecha_resultante = fecha_resultante.replace(hour=9, minute=0, second=0, microsecond=0)

        return fecha_resultante

    def extraer_duracion(self, texto):
        match = re.search(r"(\d+(?:\.\d+)?)\s*(h|horas|min|minutos)", texto.lower())
        if match:
            valor = float(match.group(1))
            if match.group(2).startswith("min"):
                return valor / 60.0
            return valor
        return 1.0  # 1 hora por defecto

    def extraer_prioridad(self, texto):
        if "alta" in texto:
            return 1
        elif "baja" in texto:
            return 3
        return 2

    def extraer_recurrencia(self, texto):
        if "diario" in texto or "cada día" in texto:
            return "diaria"
        elif "semanal" in texto or "cada semana" in texto:
            return "semanal"
        return "ninguna"

    def extraer_tipo(self, texto):
        if "estudio" in texto or "clase" in texto:
            return "estudio"
        elif "gym" in texto or "deporte" in texto:
            return "deporte"
        elif "descanso" in texto or "relaj" in texto:
            return "descanso"
        else:
            return "otro"

    # --- Agregar actividad desde lenguaje natural ---
    def agregar_actividad(self, comando):
        comando_lower = comando.lower()
        nombre = "Actividad sin nombre" # Nombre predeterminado si no se analiza nada

        # Limpieza inicial: eliminar el prefijo del comando (ej. "agregar actividad ", "nueva actividad ")
        processed_comando = comando_lower
        match_prefix = re.search(r"^(?:agregar|nueva)\s+actividad\s+", comando_lower)
        if match_prefix:
            processed_comando = comando_lower[match_prefix.end():].strip()

        # Extraer otros parámetros y eliminar sus frases del comando_procesado.
        # Esto ayuda a aislar el nombre de la actividad eliminando la sintaxis de parámetros conocida.
        # El orden de eliminación importa: eliminar patrones más largos y específicos primero.

        # --- Extraer Hora/Fecha (y eliminar frases relacionadas) ---
        # Estas expresiones regulares deben capturar las frases utilizadas en la entrada para fecha/hora
        date_time_phrases = [
            r"para\s+pasado\s+mañana", r"pasado\s+mañana",
            r"para\s+mañana", r"mañana",
            r"para\s+hoy", r"hoy",
            r"para\s+\d{4}-\d{2}-\d{2}", # ej. "para 2026-05-28"
            r"\d{4}-\d{2}-\d{2}",        # ej. "2026-05-28"
            r"a\s+las\s+\d{1,2}(?::\d{2})?(?:\s*(?:am|pm))?", # ej. "a las 16:00", "a las 4pm"
            r"\d{1,2}(?::\d{2})?(?:\s*(?:am|pm))?", # ej. "16:00", "4pm"
            r"dentro\s+de\s+\d+\s*(?:horas|minutos)" # ej. "dentro de 1 hora"
        ]
        # Ordenar por longitud, escapar caracteres especiales y añadir límites de palabra para una eliminación precisa
        date_time_phrases_regex = [r"\b" + re.escape(p).replace(r"\\s\+", r"\s+") + r"\b" for p in sorted(date_time_phrases, key=len, reverse=True)]
        for phrase_re in date_time_phrases_regex:
            processed_comando = re.sub(phrase_re, "", processed_comando).strip()
            processed_comando = re.sub(r"\s+", " ", processed_comando).strip() # Limpiar espacios dobles

        # --- Extraer Duración (y eliminar frases relacionadas) ---
        duration_phrases = [r"duracion\s+\d+(?:\.\d+)?\s*(?:h|horas|min|minutos)"]
        duration_phrases_regex = [r"\b" + re.escape(p).replace(r"\\s\+", r"\s+") + r"\b" for p in sorted(duration_phrases, key=len, reverse=True)]
        for phrase_re in duration_phrases_regex:
            processed_comando = re.sub(phrase_re, "", processed_comando).strip()
            processed_comando = re.sub(r"\s+", " ", processed_comando).strip()

        # --- Extraer Prioridad (y eliminar frases relacionadas) ---
        priority_phrases = [r"con\s+prioridad\s+(?:alta|baja|media)", r"prioridad\s+(?:alta|baja|media)"]
        priority_phrases_regex = [r"\b" + re.escape(p).replace(r"\\s\+", r"\s+") + r"\b" for p in sorted(priority_phrases, key=len, reverse=True)]
        for phrase_re in priority_phrases_regex:
            processed_comando = re.sub(phrase_re, "", processed_comando).strip()
            processed_comando = re.sub(r"\s+", " ", processed_comando).strip()

        # --- Extraer Recurrencia (y eliminar frases relacionadas) ---
        recurrence_phrases = [r"cada\s+día", r"diario", r"cada\s+semana", r"semanal"]
        recurrence_phrases_regex = [r"\b" + re.escape(p).replace(r"\\s\+", r"\s+") + r"\b" for p in sorted(recurrence_phrases, key=len, reverse=True)]
        for phrase_re in recurrence_phrases_regex:
            processed_comando = re.sub(phrase_re, "", processed_comando).strip()
            processed_comando = re.sub(r"\s+", " ", processed_comando).strip()

        # --- Extraer Tipo (y eliminar frases relacionadas) ---
        type_phrases = [r"tipo\s+(?:estudio|gym|deporte|descanso|relaj|otro)", r"estudio", r"gym", r"deporte", r"descanso", r"relaj"]
        type_phrases_regex = [r"\b" + re.escape(p).replace(r"\\s\+", r"\s+") + r"\b" for p in sorted(type_phrases, key=len, reverse=True)]
        for phrase_re in type_phrases_regex:
            processed_comando = re.sub(phrase_re, "", processed_comando).strip()
            processed_comando = re.sub(r"\s+", " ", processed_comando).strip()

        # Lo que queda en comando_procesado debe ser el nombre de la actividad
        nombre = processed_comando if processed_comando else "Actividad sin nombre"

        # Ahora, extraer los valores reales de los parámetros usando la cadena de comando completa *original*
        # El orden de estas extracciones no importa ya que buscan en la cadena completa.
        inicio = self.extraer_fecha_hora(comando_lower)
        duracion = self.extraer_duracion(comando_lower)
        prioridad = self.extraer_prioridad(comando_lower)
        recurrencia = self.extraer_recurrencia(comando_lower)
        tipo = self.extraer_tipo(comando_lower)

        nueva_inicio = inicio
        nueva_fin = nueva_inicio + timedelta(hours=duracion)

        conflictos_encontrados = self.conflictos(nueva_inicio, nueva_fin)
        warning_msg = ""
        if conflictos_encontrados:
            conflictos_str = ", ".join(conflictos_encontrados)
            warning_msg = f"⚠️ ¡CUIDADO! Esta actividad se solapa con: {conflictos_str}."

        actividad = {
            "nombre": nombre,
            "inicio": inicio.isoformat(),
            "duracion": duracion,
            "prioridad": prioridad,
            "recurrencia": recurrencia,
            "tipo": tipo
        }
        self.actividades.append(actividad)
        self.guardar()
        return_message = f"✅ Actividad '{nombre}' agregada el {inicio.strftime('%Y-%m-%d %H:%M')} por {duracion}h (prioridad {prioridad})"
        if warning_msg:
            return_message += f"\n{warning_msg}"
        return return_message

    # --- Detectar conflictos ---
    def conflictos(self, nueva_inicio, nueva_fin, omitir_id=None):
        conflictos = []
        for i, act in enumerate(self.actividades):
            if omitir_id is not None and i == omitir_id:
                continue
            a_inicio = datetime.fromisoformat(act["inicio"])
            a_fin = a_inicio + timedelta(hours=act["duracion"])
            if max(nueva_inicio, a_inicio) < min(nueva_fin, a_fin):
                conflictos.append(act["nombre"])
        return conflictos

    # --- Generar horario (ordenado por inicio) ---
    def generar_horario(self):
        # expandir recurrencias en un rango de 7 días
        hoy = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
        fin_semana = hoy + timedelta(days=7)
        eventos = []
        for act in self.actividades:
            inicio = datetime.fromisoformat(act["inicio"])
            if act["recurrencia"] == "diaria":
                fecha = inicio
                while fecha <= fin_semana:
                    eventos.append((fecha, act))
                    fecha += timedelta(days=1)
            elif act["recurrencia"] == "semanal":
                fecha = inicio
                while fecha <= fin_semana:
                    eventos.append((fecha, act))
                    fecha += timedelta(days=7)
            else:
                eventos.append((inicio, act))
        eventos.sort(key=lambda x: x[0])
        horario = []
        for fecha, act in eventos:
            fin = fecha + timedelta(hours=act["duracion"])
            horario.append({
                "actividad": act["nombre"],
                "inicio": fecha.strftime("%Y-%m-%d %H:%M"),
                "fin": fin.strftime("%Y-%m-%d %H:%M"),
                "tipo": act["tipo"]
            })
        return horario

    def mostrar_horario(self, horario):
        if not horario:
            print("📭 No hay actividades programadas.")
            return
        print("\n🗓️  HORARIO SEMANAL:")
        for bloque in horario:
            print(f"{bloque['inicio']} → {bloque['fin']} | {bloque['actividad']} ({bloque['tipo']})")

    # --- Punto de entrada principal ---
    def procesar(self, comando):
        comando = comando.lower()
        if "agregar" in comando or "nueva actividad" in comando:
            return self.agregar_actividad(comando)
        elif "horario" in comando or "mostrar" in comando:
            self.mostrar_horario(self.generar_horario())
            return "Horario generado."
        elif "conflictos" in comando:
            # La detección de conflictos ahora está integrada en agregar_actividad
            return "La detección de conflictos se realiza automáticamente al agregar una actividad."
        elif "limpiar" in comando or "borrar todo" in comando:
            return self.limpiar_actividades()
        elif "eliminar" in comando or "borrar actividad" in comando:
            # Esta expresión regular para la eliminación sigue siendo simple, considera todo después de 'eliminar actividad'
            nombre_match = re.search(r"(?:eliminar|borrar)\s+actividad\s+(.+)", comando)
            if nombre_match:
                nombre_a_eliminar = nombre_match.group(1).strip()
                return self.eliminar_actividad(nombre_a_eliminar)
            else:
                return "Por favor, especifica el nombre de la actividad a eliminar (ej: 'eliminar actividad estudiar python')."
        else:
            return "Comandos soportados: 'agregar actividad <nombre> para mañana a las 10am duración 2h prioridad alta', 'mostrar horario', 'limpiar actividades', 'eliminar actividad <nombre>'. La detección de conflictos es automática."


## Configuración de la API con Flask

Primero, necesitamos instalar Flask, un microframework para Python que nos ayudará a crear la API.

In [7]:
# Instalar Flask
!pip install Flask

In [8]:
import threading
from flask import Flask, request, jsonify

app = Flask(__name__)
agente_api = AgentePersonalAvanzado() # Instancia de nuestro agente

@app.route('/')
def home():
    return '¡Bienvenido a la API de Agente Personal! Usa los endpoints como /agregar_actividad, /mostrar_horario, etc.'

@app.route('/agregar_actividad', methods=['POST'])
def agregar_actividad_api():
    data = request.get_json()
    if not data or 'comando' not in data:
        return jsonify({'error': 'Se requiere un comando para agregar actividad'}), 400

    comando = data['comando']
    resultado = agente_api.procesar(comando)
    return jsonify({'mensaje': resultado})

@app.route('/mostrar_horario', methods=['GET'])
def mostrar_horario_api():
    horario = agente_api.generar_horario()
    return jsonify({'horario': horario})

@app.route('/limpiar_actividades', methods=['POST'])
def limpiar_actividades_api():
    resultado = agente_api.limpiar_actividades()
    return jsonify({'mensaje': resultado})

@app.route('/eliminar_actividad', methods=['POST'])
def eliminar_actividad_api():
    data = request.get_json()
    if not data or 'nombre' not in data:
        return jsonify({'error': 'Se requiere el nombre de la actividad a eliminar'}), 400

    nombre_actividad = data['nombre']
    resultado = agente_api.procesar(f"eliminar actividad {nombre_actividad}")
    return jsonify({'mensaje': resultado})

def run_api():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# Lanzar la API en un hilo separado
thread = threading.Thread(target=run_api)
thread.start()

print("✅ API corriendo en segundo plano. Puedes seguir usando el notebook.")
print("Accede a ella en http://127.0.0.1:5000")


✅ API corriendo en segundo plano. Puedes seguir usando el notebook.
Accede a ella en http://127.0.0.1:5000
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Ahora, crearemos una aplicación Flask y definiremos los *endpoints* (rutas URL) que permitirán a otras aplicaciones comunicarse con nuestro `AgentePersonalAvanzado`.

In [9]:
import requests

# Agregar una actividad
resp = requests.post("http://127.0.0.1:5000/agregar_actividad",
                     json={"comando": "agregar actividad Estudiar API para mañana a las 11am"})
print(resp.json())

# Ver horario
resp = requests.get("http://127.0.0.1:5000/mostrar_horario")
print(resp.json())

INFO:werkzeug:127.0.0.1 - - [28/May/2026 03:44:19] "POST /agregar_actividad HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/May/2026 03:44:19] "GET /mostrar_horario HTTP/1.1" 200 -


{'mensaje': "✅ Actividad 'estudiar api a las 11am' agregada el 2026-05-29 11:00 por 1.0h (prioridad 2)\n⚠️ ¡CUIDADO! Esta actividad se solapa con: estudiar api a las 11am."}
{'horario': [{'actividad': 'estudiar api a las 11am', 'fin': '2026-05-29 12:00', 'inicio': '2026-05-29 11:00', 'tipo': 'otro'}, {'actividad': 'estudiar api a las 11am', 'fin': '2026-05-29 12:00', 'inicio': '2026-05-29 11:00', 'tipo': 'otro'}]}


Para ejecutar la API dentro de Colab, usaremos `app.run()`. Ten en cuenta que si cierras esta pestaña de Colab, la API dejará de ejecutarse. Si quisieras un despliegue persistente, necesitarías un entorno de servidor dedicado.

Una vez que la celda se ejecute, verás una URL (generalmente `http://127.0.0.1:5000/`) y, si usas `ngrok` o una herramienta similar para exponer tu puerto local, podrás acceder a ella desde fuera de Colab. En este ejemplo, simplemente se ejecutará dentro del entorno de Colab.

### Interacción directa con el Agente

Aquí puedes interactuar directamente con la instancia `agente_api` que ya ha sido creada. Puedes probar comandos como:

*   `agregar actividad Estudiar Python para hoy a las 16:00 duración 1.5h prioridad alta`
*   `mostrar horario`
*   `eliminar actividad Estudiar Python`
*   `limpiar actividades`

In [10]:
# Para detener la interacción, simplemente escribe 'salir' o 'exit'.
while True:
    comando_usuario = input("Ingrese su comando para el agente (o 'salir'): ")
    if comando_usuario.lower() in ['salir', 'exit']:
        print("¡Hasta luego!")
        break

    resultado = agente_api.procesar(comando_usuario)
    print(resultado)


Ingrese su comando para el agente (o 'salir'): salir
¡Hasta luego!


### Comandos para Interactuar con la API del Agente (usando `requests`)

Aquí se muestran ejemplos de cómo interactuar con los *endpoints* de tu API personalizada utilizando la librería `requests` de Python. Asegúrate de que la celda que inicia la API (la que contiene `app.run` o `thread.start()`) esté ejecutándose.

```python
import requests
import json

# URL base de tu API, tal como se muestra al iniciar el servidor Flask
BASE_URL = "http://127.0.0.1:5000"

# --- 1. Agregar una actividad (POST /agregar_actividad) ---
# Envía un comando de lenguaje natural al agente para agregar una actividad.
# Puedes probar con diferentes fechas, horas, duraciones y prioridades.
comando_para_agregar = "agregar actividad Tarea de Álgebra para mañana a las 10:30 duración 2h prioridad alta"
resp = requests.post(f"{BASE_URL}/agregar_actividad",
                     json={"comando": comando_para_agregar})
print("\n--- Resultado al agregar actividad ---")
print(json.dumps(resp.json(), indent=2))

comando_para_agregar_2 = "agregar actividad Preparar presentación para hoy a las 15:00 duración 1h"
resp = requests.post(f"{BASE_URL}/agregar_actividad",
                     json={"comando": comando_para_agregar_2})
print("\n--- Resultado al agregar segunda actividad ---")
print(json.dumps(resp.json(), indent=2))

# --- 2. Mostrar el horario (GET /mostrar_horario) ---
# Obtiene la lista completa de actividades programadas.
resp = requests.get(f"{BASE_URL}/mostrar_horario")
print("\n--- Horario actual ---")
print(json.dumps(resp.json(), indent=2))

# --- 3. Eliminar una actividad (POST /eliminar_actividad) ---
# Elimina una actividad por su nombre exacto.
nombre_para_eliminar = "Tarea de Álgebra"
resp = requests.post(f"{BASE_URL}/eliminar_actividad",
                     json={"nombre": nombre_para_eliminar})
print(f"\n--- Resultado al eliminar '{nombre_para_eliminar}' ---")
print(json.dumps(resp.json(), indent=2))

# Vuelve a mostrar el horario para verificar la eliminación
resp = requests.get(f"{BASE_URL}/mostrar_horario")
print("\n--- Horario después de eliminar ---")
print(json.dumps(resp.json(), indent=2))

# --- 4. Limpiar todas las actividades (POST /limpiar_actividades) ---
# ¡ATENCIÓN! Esto eliminará todas las actividades de tu agente.
# Descomenta las siguientes líneas para usar esta función.
# resp = requests.post(f"{BASE_URL}/limpiar_actividades")
# print("\n--- Resultado al limpiar actividades ---")
# print(json.dumps(resp.json(), indent=2))
# resp = requests.get(f"{BASE_URL}/mostrar_horario")
# print("\n--- Horario después de limpiar (debería estar vacío) ---")
# print(json.dumps(resp.json(), indent=2))
```

---